# Implement Beam Search for LLM Decoding

**Difficulty**: 🟡 Medium

### Problem Statement

Beam search is a deterministic decoding algorithm that maintains multiple candidate sequences (beams) at each time step. Instead of greedily picking the single best token, beam search expands each beam by the entire vocabulary, scores all candidates by cumulative log-probability, and keeps only the top `beam_width` sequences. This explores a broader search space and often produces higher-quality outputs than greedy decoding.

Your goal is to implement beam search from scratch using a simple dummy language model (random logits) for testing purposes.

---

### Requirements

1. **Initialization**
   - Start with a single beam containing just the `start_token`.
   - Initialize its cumulative log-probability to 0.0.

2. **Expansion**
   - At each step, feed each beam's last token into the model to get logits.
   - Convert logits to log-probabilities using `log_softmax`.
   - For each beam, create `vocab_size` candidate extensions, each with an updated cumulative score.

3. **Pruning**
   - From all candidates across all beams, keep only the top `beam_width` by cumulative log-probability.

4. **Output**
   - After `max_len` steps, return the top beam (highest cumulative score) as the output sequence.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Use a dummy model that returns random logits for any input token.
- ✅ Output sequences should have length `max_len`.
- ✅ The top beam should have the highest cumulative score among all beams.

---

**Companies**: Google, DeepMind, Meta, Apple

---

<details>
  <summary>💡 Hint</summary>

  - Represent each beam as a tuple: `(sequence_list, cumulative_log_prob)`.
  - At each step, collect all candidates into a single list, sort by score, and take the top `beam_width`.
  - Use `F.log_softmax(logits, dim=-1)` to convert model output to log-probabilities.
  - To make it efficient, you can use `torch.topk` on each beam's log-probs instead of iterating over the full vocabulary.
  - The dummy model can simply be a `nn.Linear` or even just `torch.randn` — the algorithm is the same regardless.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Data generation / setup
torch.manual_seed(42)
vocab_size = 50257

# Dummy language model: given any token, returns random logits over the vocabulary.
# In practice this would be a real transformer, but for testing beam search
# the algorithm is the same.
class DummyLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed = nn.Embedding(vocab_size, 64)
        self.head = nn.Linear(64, vocab_size)
    
    def forward(self, token_id):
        """Takes a single token id (int) and returns logits of shape (1, vocab_size)."""
        x = self.embed(torch.tensor([token_id]))
        return self.head(x)

model = DummyLM(vocab_size)
start_token = 0  # Beginning-of-sequence token
print(f"Dummy model created with vocab_size={vocab_size}")
print(f"Sample output shape: {model(start_token).shape}")

In [ ]:
def beam_search(model, start_token, beam_width=5, max_len=20):
    """
    Perform beam search decoding.
    
    Args:
        model: A language model that takes a token id and returns logits (1, vocab_size)
        start_token (int): The starting token id
        beam_width (int): Number of beams to maintain
        max_len (int): Maximum sequence length to generate
        
    Returns:
        list: The best sequence (list of token ids) of length max_len
    """
    # Step 1: Initialize beams
    # Each beam is (sequence, cumulative_log_prob)
    # TODO: Start with a single beam containing just start_token with score 0.0
    ...
    
    # Step 2: Iterate for max_len - 1 steps (start_token is already the first token)
    for step in range(max_len - 1):
        # TODO: Collect all candidate expansions
        # For each beam:
        #   - Get logits from model for the last token in the sequence
        #   - Convert to log probabilities
        #   - For top candidates, create new beams with extended sequence and updated score
        ...
        
        # TODO: Prune to keep only top beam_width candidates
        ...
    
    # Step 3: Return the best beam (highest cumulative log-probability)
    # TODO: Sort beams by score and return the best sequence
    ...

In [ ]:
# Validation
print("=== Beam Search Validation ===")
print()

beam_width = 5
max_len = 20

# Test 1: Output sequence should have the correct length
result = beam_search(model, start_token, beam_width=beam_width, max_len=max_len)
assert isinstance(result, list), f"Output should be a list, got {type(result)}"
assert len(result) == max_len, f"Sequence length should be {max_len}, got {len(result)}"
print(f"Test 1 PASSED: Output sequence has length {len(result)}")
print(f"  Sequence: {result[:10]}... (first 10 tokens)")

# Test 2: First token should be start_token
assert result[0] == start_token, f"First token should be {start_token}, got {result[0]}"
print(f"Test 2 PASSED: First token is start_token ({start_token})")

# Test 3: All tokens should be valid vocabulary indices
assert all(0 <= t < vocab_size for t in result), "All tokens should be valid vocab indices"
print(f"Test 3 PASSED: All tokens are valid indices in [0, {vocab_size})")

# Test 4: Beam search with beam_width=1 should equal greedy decoding
greedy_result = beam_search(model, start_token, beam_width=1, max_len=10)
# Verify greedy by running manually
greedy_manual = [start_token]
for _ in range(9):
    with torch.no_grad():
        logits = model(greedy_manual[-1])
    next_token = logits.argmax(dim=-1).item()
    greedy_manual.append(next_token)
assert greedy_result == greedy_manual, \
    f"beam_width=1 should match greedy decoding.\nBeam: {greedy_result}\nGreedy: {greedy_manual}"
print(f"Test 4 PASSED: beam_width=1 matches greedy decoding")

# Test 5: Different beam widths should work
for bw in [2, 3, 10]:
    res = beam_search(model, start_token, beam_width=bw, max_len=10)
    assert len(res) == 10
    print(f"Test 5: beam_width={bw} produced sequence of length {len(res)}")

print("\nAll tests passed!")